# Tracefold News 学习闭环：固定 24 小时证据复算

```yaml
channel: C  # committed snapshot: notebooks/snapshots/, offline and recomputable from the repository alone
purpose: "复算 2026-08-21 固定 24 小时窗口的漏斗、版本混合、读者分布与价格观察覆盖率，避免把移动的 `--hours 24` 或事后涨跌误当成学习真值。"
window: "news_events.opened_at_ms ∈ [2026-08-20T02:30:09.757Z, 2026-08-21T02:30:09.757Z)，由快照自身的 event_window 字段固化，不由执行时刻决定。"
identity: "快照 provenance 固化 handoff_git_sha dedd869f、migration 20260820_0283、news_triage_prompt_v9 / news_triage_policy_v6 及其 SHA-256；四个 cohort 的混合正是本文要说明的对象。"
safety: "只读 notebooks/snapshots/ 内的聚合 JSON；不连生产库，不含 provider 原文、中文卡片或凭据。"
```

产生该快照的只读 SQL 与它配对提交在同一目录；输出即证据，随文件一起提交。

In [1]:
import json
import math
from pathlib import Path

snapshot_path = Path("snapshots/news-review-24h-audit-snapshot-2026-08-21.json")
if not snapshot_path.exists():
    snapshot_path = Path("notebooks") / snapshot_path
data = json.loads(snapshot_path.read_text(encoding="utf-8"))
data["artifact"], data["event_window"], data["provenance"]["handoff_git_sha"]

('tracefold_news_review_24h_audit_v1',
 {'field': 'news_events.opened_at_ms',
  'start_inclusive_ms': 1787193009757,
  'end_exclusive_ms': 1787279409757,
  'start_taipei': '2026-08-20T10:30:09.757+08:00',
  'end_taipei': '2026-08-21T10:30:09.757+08:00'},
 'dedd869f93502bdad4bb219100e56dbb57568f11')

## 1. 漏斗一致性

一个基础校验是：进入 Triage 的 Event 必须能被四种最终决定完整解释；最终 `push + escalate` 也应等于本窗口最终查询到的 sent Events。

In [2]:
f = data["funnel"]
assert f["triaged"] == f["push"] + f["escalate"] + f["drop"] + f["throttled"]  # noqa: S101
assert f["opened_window_events_with_final_first_delivery_sent"] == f["push"] + f["escalate"]  # noqa: S101
funnel = [
    {"stage": "Events", "count": f["events"], "share_of_events": 1.0},
    {"stage": "Triaged", "count": f["triaged"], "share_of_events": f["triaged"] / f["events"]},
    {
        "stage": "Sent",
        "count": f["opened_window_events_with_final_first_delivery_sent"],
        "share_of_events": f["opened_window_events_with_final_first_delivery_sent"] / f["events"],
    },
    {"stage": "Drop", "count": f["drop"], "share_of_events": f["drop"] / f["events"]},
    {"stage": "Throttled", "count": f["throttled"], "share_of_events": f["throttled"] / f["events"]},
]
print(json.dumps(funnel, ensure_ascii=False, indent=2))

[
  {
    "stage": "Events",
    "count": 1628,
    "share_of_events": 1.0
  },
  {
    "stage": "Triaged",
    "count": 1572,
    "share_of_events": 0.9656019656019657
  },
  {
    "stage": "Sent",
    "count": 419,
    "share_of_events": 0.2573710073710074
  },
  {
    "stage": "Drop",
    "count": 1065,
    "share_of_events": 0.6541769041769042
  },
  {
    "stage": "Throttled",
    "count": 88,
    "share_of_events": 0.05405405405405406
  }
]


## 2. 版本混合

同一 24 小时里存在四个 Prompt/Policy 组合。因此任何不分 cohort 的总体百分比都不能归因给“当前 Agent”。下面的图只描述组成，不比较优劣。

In [3]:
cohorts = []
for row in data["cohorts"]:
    item = dict(row)
    item["cohort"] = f"{row['policy'].rsplit('_', 1)[-1]} / {row['prompt'].rsplit('_', 1)[-1]}"
    item["push_share"] = row["pushed"] / row["events"]
    cohorts.append(item)
print(json.dumps(cohorts, ensure_ascii=False, indent=2))

[
  {
    "policy": "news_triage_policy_v4",
    "prompt": "news_triage_prompt_v8",
    "events": 220,
    "pushed": 57,
    "throttled": 19,
    "cohort": "v4 / v8",
    "push_share": 0.2590909090909091
  },
  {
    "policy": "news_triage_policy_v5",
    "prompt": "news_triage_prompt_v8",
    "events": 702,
    "pushed": 181,
    "throttled": 42,
    "cohort": "v5 / v8",
    "push_share": 0.25783475783475784
  },
  {
    "policy": "news_triage_policy_v6",
    "prompt": "news_triage_prompt_v8",
    "events": 34,
    "pushed": 8,
    "throttled": 1,
    "cohort": "v6 / v8",
    "push_share": 0.23529411764705882
  },
  {
    "policy": "news_triage_policy_v6",
    "prompt": "news_triage_prompt_v9",
    "events": 616,
    "pushed": 173,
    "throttled": 26,
    "cohort": "v6 / v9",
    "push_share": 0.28084415584415584
  }
]


In [4]:
scale = 40 / max(row["pushed"] + row["throttled"] for row in cohorts)
for row in cohorts:
    sent_bar = "█" * round(row["pushed"] * scale)
    throttled_bar = "░" * round(row["throttled"] * scale)
    print(f"{row['cohort']:9} {sent_bar}{throttled_bar} sent={row['pushed']} throttled={row['throttled']}")

v4 / v8   ██████████░░░ sent=57 throttled=19
v5 / v8   ████████████████████████████████░░░░░░░░ sent=181 throttled=42
v6 / v8   █ sent=8 throttled=1
v6 / v9   ███████████████████████████████░░░░░ sent=173 throttled=26


## 3. 推送重点分布

这只能说明读者收到了什么，不说明这些卡片质量高。人工标签为零时，不能计算 precision、recall 或 false-push rate。

In [5]:
sent_total = sum(row["pushed"] for row in data["audience"])
audience = [{**row, "share_of_sent": row["pushed"] / sent_total} for row in data["audience"]]
print(json.dumps(audience, ensure_ascii=False, indent=2))

[
  {
    "audience": "crypto",
    "triaged": 427,
    "pushed": 174,
    "throttled": 30,
    "share_of_sent": 0.4152744630071599
  },
  {
    "audience": "macro",
    "triaged": 475,
    "pushed": 147,
    "throttled": 45,
    "share_of_sent": 0.35083532219570407
  },
  {
    "audience": "us_equity",
    "triaged": 191,
    "pushed": 98,
    "throttled": 13,
    "share_of_sent": 0.23389021479713604
  },
  {
    "audience": "none",
    "triaged": 479,
    "pushed": 0,
    "throttled": 0,
    "share_of_sent": 0.0
  }
]


## 4. 市场反应只是覆盖有限的旁证

`directional hit` 只问“模型写 bullish/bearish 后，1 小时 raw return 符号是否相同”。它没有市场/行业基线，没有排除共同事件，也没有证明新闻造成价格变化。这里复算它只是为了说明为何不能把它当 reward。

In [6]:
def wilson(successes, n, z=1.96):
    if n == 0:
        return (math.nan, math.nan)
    p = successes / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centre - half, centre + half


rows = []
for cohort, values in data["one_hour_reaction"].items():
    if not isinstance(values, dict):
        continue
    lo, hi = wilson(values["hit"], values["directional"])
    rows.append(
        {
            "cohort": cohort,
            "mature_eligible": values["eligible"],
            "priced": values["priced"],
            "coverage": values["priced"] / values["eligible"],
            "directional_n": values["directional"],
            "sign_alignment": values["hit"] / values["directional"],
            "wilson_95_low": lo,
            "wilson_95_high": hi,
        }
    )
reaction = rows
print(json.dumps(reaction, ensure_ascii=False, indent=2))

[
  {
    "cohort": "delivered",
    "mature_eligible": 402,
    "priced": 253,
    "coverage": 0.6293532338308457,
    "directional_n": 241,
    "sign_alignment": 0.5020746887966805,
    "wilson_95_low": 0.43941251453622654,
    "wilson_95_high": 0.5646717587245266
  },
  {
    "cohort": "held",
    "mature_eligible": 1109,
    "priced": 247,
    "coverage": 0.22272317403065825,
    "directional_n": 185,
    "sign_alignment": 0.5027027027027027,
    "wilson_95_low": 0.4313343411435916,
    "wilson_95_high": 0.5739611022332914
  }
]


In [7]:
assert data["feedback"]["operator_labels_all_time"] == 0  # noqa: S101
assert data["feedback"]["precision_recall_status"] == "not_identifiable"  # noqa: S101
by_cohort = {row["cohort"]: row for row in reaction}
difference = by_cohort["delivered"]["sign_alignment"] - by_cohort["held"]["sign_alignment"]
print(f"Delivered minus held raw-sign alignment: {difference:.4%}")
print("This is descriptive only; it is not a causal quality estimate.")

Delivered minus held raw-sign alignment: -0.0628%
This is descriptive only; it is not a causal quality estimate.


## 5. 可用于决策的结论

1. 在线管道的完整性是健康的；学习平面没有人工真值。
2. 版本混合使总体 24h 指标无法评价当前 Prompt。
3. sent 与 held 的 1h raw-sign alignment 几乎相同，但这既不能证明 Agent 无用，也不能证明任何新闻与价格有因果关系。
4. 下一步数据资产应是不可变 evidence snapshot、多维人工 rubric、fact-cluster 与 temporal holdout，而不是把价格涨跌写成 reward。

固定窗口 SQL 与完整方法见 `snapshots/news-review-24h-audit-2026-08-21.sql`；架构解释见 `../docs/research/news-review-architecture-audit-2026-08-21.md`。